# Range generation

Want to first build a data pipeline for ranges, including the base features. Will later build a better generator all types of appliances


In [52]:
import random
from faker import Faker

fake = Faker()

range_types = ["gas", "electric", "induction", "dual fuel"] #to start the generation

#this keeps it realistic, some brands dont make certain types of ranges
brands_by_type = {
    "gas": ["Frigidaire", "GE", "Whirlpool", "LG", "Samsung", "KitchenAid", "Wolf"],
    "electric": ["Frigidaire", "GE", "Whirlpool", "LG", "Samsung", "Bosch"],
    "induction": ["GE", "Samsung", "Bosch", "Cafe", "Miele", "Fisher & Paykel"],
    "dual fuel": ["KitchenAid", "Cafe", "Thermador", "Wolf", "Monogram"],
}

#to account for possibly more features in a more premium product
brand_tiers = {
    "budget": ["Frigidaire", "Whirlpool"],
    "mainstream": ["GE", "LG", "Samsung"],
    "premium": ["KitchenAid", "Bosch", "Cafe"],
    "luxury": ["Miele", "Fisher & Paykel", "Thermador", "Wolf", "Monogram"],
}

# googled avg price, took median of the midrange prices
base_price_by_type = {
    "electric": 1200,
    "gas": 1600,
    "induction": 2400,
    "dual fuel": 3250,
}

#to account for price inflation based off the tier
tier_multiplier = {
    "budget": 0.85,
    "mainstream": 1.0,
    "premium": 1.45,
    "luxury": 2.4,
}

#the better the tier, the more features
tier_feature_count = {
    "budget": [1, 3],
    "mainstream": [2, 4],
    "premium": [2, 5],
    "luxury": [4, 7],
}

#to get the brand tier - affects the number of features, price and whatnot
def get_brand_tier(brand):
    for tier in brand_tiers.keys():
        if brand in brand_tiers[tier]:
            return tier
    return None

#actual range generation funciton
def generate_range():

    #choose some stuff, the brand and brand tier depend on the range type, same with features
    range_type = random.choice(range_types)

    brand = random.choice(brands_by_type[range_type])
    brand_tier = get_brand_tier(brand)

    width = random.choices([24, 30, 36, 48, 60], weights=[20, 45, 25, 8, 2], k=1)[0]

    depth = random.choice([25, 26, 27, 28, 29, 30, 31])

    height = random.choices([35, 36, 37, 38, 39, 40, 42, 44, 46], weights=[8, 35, 15, 12, 8, 8, 6, 5, 3],k=1)[0]

    burners = random.choice([4, 5, 6])
    features = random.sample(
        ["Air Fry", "Convection", "Self Clean", "WiFi", "Griddle", "Steam Clean", "Smart enabled"],
        k=random.randint(tier_feature_count[brand_tier][0], tier_feature_count[brand_tier][1]))
    finish = random.choice(["Stainless Steel", "Black Stainless", "White", "Black"])

    # somewhat accurate price generation
    price = base_price_by_type[range_type]
    price *= tier_multiplier[brand_tier]

    # width should affect price the most
    if width == 24:
        price *= 0.85
    elif width == 30:
        price *= 1.0
    elif width == 36:
        price *= 1.35
    elif width == 48:
        price *= 2.1
    elif width == 60:
        price *= 3.0

    # depth and height affect price, but less than width
    price *= 1 + ((depth - 26) * 0.03)
    price *= 1 + ((height - 36) * 0.015)

    # more burners usually means higher price
    if burners == 5:
        price *= 1.08
    elif burners == 6:
        price *= 1.18

    # features still add value
    price += len(features) * 150

    # adding some variance in price
    price *= random.uniform(0.9, 1.1)
    price = round(price, -1)

    #pretty random ratings - probably will try to push the higher rated stuff....
    rating = round(random.normalvariate(4.2, 0.25), 1)
    rating = min(max(rating, 3.2), 5.0)

    return {
        "product_id": fake.unique.bothify("RNG-#####"),
        "category": "range",
        "range_type": range_type,
        "brand": brand,
        "brand_tier": brand_tier,
        "width_inches": width,
        "height_inches": height,
        "depth_inches": depth,
        "burners": burners,
        "finish": finish,
        "features": features,
        "price": int(price),
        "rating": rating,
    }


print(generate_range())


{'product_id': 'RNG-72244', 'category': 'range', 'range_type': 'electric', 'brand': 'Frigidaire', 'brand_tier': 'budget', 'width_inches': 48, 'height_inches': 36, 'depth_inches': 29, 'burners': 6, 'finish': 'Stainless Steel', 'features': ['Convection'], 'price': 2630, 'rating': 4.0}
